# Exploratory data analysis

Compare the original class frequencies, amount and time distributions, anonymized components, and IQR flags. EDA here describes the source data before exclusions.

Run the training CLI before executing this notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'configs/full.json').is_file())
sys.path.insert(0, str(root))
from src.utils import read_json

pointer = root / 'models/latest.json'
if not pointer.is_file():
    raise FileNotFoundError('Run python main.py --mode train from the project root first.')
run = root / 'reports/runs' / read_json(pointer)['run_id']
metadata = read_json(run / 'metadata.json')
print(metadata['profile'], metadata['run_id'])

In [ ]:
summary = read_json(run / 'dataset.json')
print(f"{summary['rows']:,} records; {summary['fraud']:,} fraud ({summary['fraud_percent']:.4f}%).")
print(f"Exact duplicates: {summary['duplicate_rows']:,}; missing cells: {sum(summary['missing'].values()):,}.")
display(pd.read_csv(run / 'preview.csv'))
display(pd.DataFrame({'dtype': summary['dtypes'], 'missing': summary['missing']}))
display(pd.read_csv(run / 'class_statistics.csv').head(12))
display(pd.read_csv(run / 'summary.csv').head())
display(pd.read_csv(run / 'outliers.csv').sort_values('outside_1_5_iqr', ascending=False).head())
for name in ['class_distribution', 'amount_time', 'class_features', 'correlation']:
    display(Image(filename=str(run / 'figures' / f'{name}.png')))
print('A skewed amount distribution motivates a log axis for visualization. IQR flags do not trigger deletion.')